# Isaac Patch False-Positive Demo Notebook

This is a cleaned notebook extracted from `Isaac_patch.ipynb`.

It does **not** train a patch and does **not** run full dataset evaluation. It only:

1. mounts Google Drive,
2. loads the YOLO model from Drive,
3. loads the adversarial patch tensor from Drive,
4. creates **5 FP demo images** by overlaying that patch onto clean images,
5. creates **5 FP demo images** from already-patched images loaded from Drive,
6. uses the same strict false-positive logic from the original notebook:
   - class-aware TP matching at IoU ≥ `GT_IOU_MATCH`,
   - unmatched predictions close to any GT are ignored,
   - only remaining predictions are drawn as strict FPs.

Outputs are saved under `OUT_ROOT` in Drive.


In [ ]:
# Install + mount Drive
!pip -q install ultralytics

from google.colab import drive
drive.mount('/content/drive')


## Configuration

Check these paths before running. I kept the paths that appeared in your original notebook and grouped them by purpose.

- **Overlay method:** starts from clean images and overlays `PATCH_PATH` on the fly.
- **Loaded-patched-image method:** loads images that already contain the patch from Drive and only draws FPs.


In [ ]:
from pathlib import Path

# -------------------------
# Main Drive paths
# -------------------------
MODEL_PATH = Path('/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt')

# Patch tensor used for the on-the-fly overlay method.
# This is the path used in your dataset-creation cell.
PATCH_PATH = Path('/content/drive/MyDrive/yolo_patch/Billboard1/patches/best_bill1.pt')

# Clean dataset used for the overlay method.
OVERLAY_CLEAN_ROOT = Path('/content/drive/MyDrive/isaac_scene1/clean/yolo_filtered')
OVERLAY_SPLIT = 'test'

# Already-patched dataset used for the loaded-patched-image method.
# Your original FP cells used this root:
LOADED_PATCHED_ROOT = Path('/content/drive/MyDrive/isaac patched/yolo_filtered')
LOADED_SPLIT = 'train'

# Alternative already-patched dataset produced by your patch-creation cell.
# If you want to use it instead, uncomment this line:
# LOADED_PATCHED_ROOT = Path('/content/drive/MyDrive/isaac_scene1/transf_Patch1'); LOADED_SPLIT = 'test'

# Output folder in Drive
OUT_ROOT = Path('/content/drive/MyDrive/isaac_fp_demo_outputs')

# -------------------------
# Demo controls
# -------------------------
N_DEMOS = 5
REQUIRE_AT_LEAST_ONE_FP = True   # tries to save images that actually have strict FPs
MAX_SCAN_IMAGES = 300            # avoids scanning a huge split forever

# If None, the notebook auto-selects images.
# To force specific images, use for example:
# DEMO_IMAGE_NAMES = ['rgb_0067.png', 'rgb_0123.png', 'rgb_0180.png', 'rgb_0250.png', 'rgb_0300.png']
DEMO_IMAGE_NAMES = None

# -------------------------
# YOLO / FP settings copied from your FP visualization cells
# -------------------------
IMGSZ = 640
CONF = 0.25
IOU = 0.6
MAX_DET = 1000

GT_IOU_MATCH = 0.5
IGNORE_IF_OVERLAPS_ANY_GT_IOU = 0.3

# Patch placement copied from your patched-dataset creation cell
X_OFFSET = 400
Y_OFFSET = 270

OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Outputs will be saved to:', OUT_ROOT)


## Helper functions

These are the only functions kept from the original notebook: dataset path detection, patch loading/overlay, YOLO-label reading, IoU, strict FP filtering, and FP drawing.


In [ ]:

import math
import os
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from ultralytics import YOLO

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

demo_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMGSZ, IMGSZ)),
])


def must_exist(path: Path, what: str):
    if not path.exists():
        raise FileNotFoundError(f'Missing {what}: {path}')


def resolve_yolo_dirs(root: Path, split: str):
    """Supports both ROOT/images/split + ROOT/labels/split and ROOT/split/images + ROOT/split/labels."""
    root = Path(root)

    layout_a_img = root / 'images' / split
    layout_a_lab = root / 'labels' / split
    if layout_a_img.exists():
        return layout_a_img, layout_a_lab

    layout_b_img = root / split / 'images'
    layout_b_lab = root / split / 'labels'
    if layout_b_img.exists():
        return layout_b_img, layout_b_lab

    raise FileNotFoundError(
        f'Could not find images for root={root}, split={split}. Expected either:\n'
        f'  {layout_a_img}\n'
        f'or\n'
        f'  {layout_b_img}'
    )


def list_images(img_dir: Path):
    return sorted([p for p in Path(img_dir).iterdir() if p.suffix.lower() in IMAGE_EXTS])


def choose_image_paths(img_dir: Path, n: int, preferred_names=None, max_scan=None):
    all_paths = list_images(img_dir)
    if not all_paths:
        raise RuntimeError(f'No images found in {img_dir}')

    by_name = {p.name: p for p in all_paths}
    chosen = []

    if preferred_names:
        missing = []
        for name in preferred_names:
            if name in by_name:
                chosen.append(by_name[name])
            else:
                missing.append(name)
        if missing:
            print(f'Warning: {len(missing)} requested image(s) not found in {img_dir}: {missing[:10]}')

    # Fill remaining slots from the split.
    used = {p.name for p in chosen}
    for p in all_paths:
        if len(chosen) >= n:
            break
        if p.name not in used:
            chosen.append(p)

    # For FP search, we may scan more than n.
    if max_scan is not None:
        scan_paths = []
        used = set()
        for p in chosen:
            if p.name not in used:
                scan_paths.append(p)
                used.add(p.name)
        for p in all_paths:
            if len(scan_paths) >= max_scan:
                break
            if p.name not in used:
                scan_paths.append(p)
                used.add(p.name)
        return scan_paths

    return chosen[:n]


def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []
    lines = txt_path.read_text(encoding='utf-8', errors='ignore').strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue
    return out


def xywhn_to_xyxy(x, y, w, h, W, H):
    x1 = (x - w / 2.0) * W
    y1 = (y - h / 2.0) * H
    x2 = (x + w / 2.0) * W
    y2 = (y + h / 2.0) * H
    return [x1, y1, x2, y2]


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def best_iou_against_all_gt(box, all_gt_boxes):
    best_iou = 0.0
    best_cls = None
    for cls_id, gbox in all_gt_boxes:
        iouv = iou_xyxy(box, gbox)
        if iouv > best_iou:
            best_iou = iouv
            best_cls = cls_id
    return best_iou, best_cls


def cls_name(class_names, c):
    if c is None:
        return 'none'
    if isinstance(class_names, dict):
        return class_names.get(int(c), str(c))
    return class_names[int(c)] if 0 <= int(c) < len(class_names) else str(c)


def get_strict_fp_boxes(result, gt_by_class, gt_iou_match=0.5, ignore_if_overlaps_any_gt_iou=0.3):
    """
    Same logic as your strict FP cells:

    Stage 1: class-aware greedy matching. Correct class + IoU >= gt_iou_match is TP.
    Stage 2: unmatched predictions that overlap any GT by ignore_if_overlaps_any_gt_iou are ignored.
    Stage 3: remaining unmatched predictions are strict false positives.
    """
    tp_boxes = []
    ignored_boxes = []
    strict_fp_boxes = []

    if result.boxes is None or len(result.boxes) == 0:
        return strict_fp_boxes, tp_boxes, ignored_boxes

    pred_cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    pred_conf = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    pred_xyxy = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)

    preds = [(int(pred_cls[i]), pred_xyxy[i].tolist(), float(pred_conf[i])) for i in range(len(pred_cls))]
    preds.sort(key=lambda z: z[2], reverse=True)

    remaining = {c: list(lst) for c, lst in gt_by_class.items()}
    all_gt_boxes = [(gt_cls, b) for gt_cls, boxes in gt_by_class.items() for b in boxes]

    unmatched_preds = []

    for c, pbox, conf in preds:
        best_iou = 0.0
        best_j = None
        gtl = remaining.get(c, [])

        for j, gbox in enumerate(gtl):
            iouv = iou_xyxy(pbox, gbox)
            if iouv > best_iou:
                best_iou = iouv
                best_j = j

        if best_j is not None and best_iou >= gt_iou_match:
            tp_boxes.append((c, pbox, conf, best_iou))
            gtl.pop(best_j)
            remaining[c] = gtl
        else:
            unmatched_preds.append((c, pbox, conf, best_iou))

    for c, pbox, conf, best_same_class_iou in unmatched_preds:
        best_any_iou, best_any_cls = best_iou_against_all_gt(pbox, all_gt_boxes)
        if best_any_iou >= ignore_if_overlaps_any_gt_iou:
            ignored_boxes.append((c, pbox, conf, best_any_iou, best_any_cls))
        else:
            strict_fp_boxes.append((c, pbox, conf, best_any_iou, best_any_cls))

    return strict_fp_boxes, tp_boxes, ignored_boxes


def draw_strict_fp_boxes(img_rgb, strict_fp_boxes, class_names, title=None):
    out = cv2.cvtColor(np.array(img_rgb), cv2.COLOR_RGB2BGR)

    for cls_id, box, conf, best_iou, best_any_cls in strict_fp_boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

        pred_name = cls_name(class_names, cls_id)
        gt_name = cls_name(class_names, best_any_cls)
        label = f'FP {pred_name} {conf:.2f} IoU {best_iou:.2f} vs {gt_name}'

        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
        y_text = max(18, y1)
        cv2.rectangle(out, (x1, y_text - th - 6), (x1 + tw + 4, y_text), (0, 0, 255), -1)
        cv2.putText(out, label, (x1 + 2, y_text - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

    if title:
        # Black title strip at top for saved demo images.
        cv2.rectangle(out, (0, 0), (out.shape[1], 26), (0, 0, 0), -1)
        cv2.putText(out, title[:120], (8, 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1, cv2.LINE_AA)

    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)


def load_patch_tensor(patch_path: Path):
    obj = torch.load(str(patch_path), map_location='cpu')

    if isinstance(obj, dict):
        # Common possible keys if the patch was saved inside a dict.
        for key in ['patch', 'best_patch', 'tensor', 'image']:
            if key in obj:
                obj = obj[key]
                break
        else:
            raise ValueError(f'Patch file is a dict but no patch-like key was found. Keys: {list(obj.keys())[:20]}')

    if isinstance(obj, (list, tuple)):
        obj = obj[0]

    if not torch.is_tensor(obj):
        obj = torch.as_tensor(obj)

    patch = obj.detach().float().cpu()

    if patch.ndim == 4:
        patch = patch[0]

    # Accept CHW or HWC.
    if patch.ndim != 3:
        raise ValueError(f'Expected patch tensor with shape CxHxW or HxWxC, got {tuple(patch.shape)}')

    if patch.shape[0] not in [1, 3, 4] and patch.shape[-1] in [1, 3, 4]:
        patch = patch.permute(2, 0, 1)

    if patch.shape[0] == 4:
        patch = patch[:3]
    if patch.shape[0] == 1:
        patch = patch.repeat(3, 1, 1)

    if patch.max().item() > 1.0:
        patch = patch / 255.0

    return patch.clamp(0, 1)


def overlay_patch(image_tensor, patch_tensor, x_offset=X_OFFSET, y_offset=Y_OFFSET):
    patched = image_tensor.clone()
    _, H, W = patched.shape
    _, ph, pw = patch_tensor.shape

    if ph > H or pw > W:
        raise ValueError(f'Patch is larger than the image: image=({H},{W}), patch=({ph},{pw})')

    # Keep the requested position, but clamp if it would go outside.
    x = int(min(max(x_offset, 0), W - pw))
    y = int(min(max(y_offset, 0), H - ph))

    patched[:, y:y+ph, x:x+pw] = patch_tensor
    return patched, (x, y)


def tensor_to_uint8_rgb(t):
    return (t.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.uint8)


def load_image_tensor(path: Path, device):
    pil = Image.open(path).convert('RGB')
    return demo_transform(pil).to(device)


def build_gt_by_class(label_path: Path, H=IMGSZ, W=IMGSZ):
    gt = read_yolo_gt_labels(label_path)
    gt_by_class = {}
    for c, x, y, w, h in gt:
        gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h, W, H))
    return gt, gt_by_class


def predict_and_draw_fp(model, image_tensor, label_path, class_names, title):
    results = model.predict(
        source=image_tensor.unsqueeze(0),
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        max_det=MAX_DET,
        device=device,
        verbose=False,
    )
    r = results[0]

    gt, gt_by_class = build_gt_by_class(label_path)
    strict_fp_boxes, tp_boxes, ignored_boxes = get_strict_fp_boxes(
        r,
        gt_by_class,
        gt_iou_match=GT_IOU_MATCH,
        ignore_if_overlaps_any_gt_iou=IGNORE_IF_OVERLAPS_ANY_GT_IOU,
    )

    img_rgb = tensor_to_uint8_rgb(image_tensor)
    vis_rgb = draw_strict_fp_boxes(img_rgb, strict_fp_boxes, class_names, title=title)

    stats = {
        'gt': len(gt),
        'preds': 0 if r.boxes is None else len(r.boxes),
        'tp': len(tp_boxes),
        'strict_fp': len(strict_fp_boxes),
        'ignored': len(ignored_boxes),
    }
    return vis_rgb, stats


def save_preview_grid(rows, out_path: Path, title: str):
    if not rows:
        print(f'No rows to preview for {title}')
        return

    cols = min(5, len(rows))
    rows_n = math.ceil(len(rows) / cols)
    plt.figure(figsize=(4.5 * cols, 4.8 * rows_n))

    for i, row in enumerate(rows, 1):
        img = Image.open(row['out_path']).convert('RGB')
        plt.subplot(rows_n, cols, i)
        plt.imshow(img)
        plt.title(f"{row['image']} | FP={row['strict_fp']}", fontsize=9)
        plt.axis('off')

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(out_path, dpi=160, bbox_inches='tight')
    plt.show()
    print('Saved preview grid:', out_path)


## Load model, patch, and validate Drive paths

In [ ]:
must_exist(MODEL_PATH, 'YOLO model')
must_exist(PATCH_PATH, 'patch tensor')
must_exist(OVERLAY_CLEAN_ROOT, 'overlay clean dataset root')
must_exist(LOADED_PATCHED_ROOT, 'loaded patched dataset root')

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

model = YOLO(str(MODEL_PATH))
try:
    model.to(device)
except Exception as e:
    print('Warning: model.to(device) failed, prediction will still try to use device argument:', e)

class_names = model.names
patch_cpu = load_patch_tensor(PATCH_PATH)
patch = patch_cpu.to(device)

print('Model:', MODEL_PATH)
print('Patch:', PATCH_PATH)
print('Patch shape:', tuple(patch.shape), '| min/max:', float(patch.min()), float(patch.max()))

overlay_img_dir, overlay_lab_dir = resolve_yolo_dirs(OVERLAY_CLEAN_ROOT, OVERLAY_SPLIT)
loaded_img_dir, loaded_lab_dir = resolve_yolo_dirs(LOADED_PATCHED_ROOT, LOADED_SPLIT)

print('\nOverlay clean images:', overlay_img_dir)
print('Overlay labels:', overlay_lab_dir, '| exists:', overlay_lab_dir.exists())
print('Loaded patched images:', loaded_img_dir)
print('Loaded labels:', loaded_lab_dir, '| exists:', loaded_lab_dir.exists())


## Method 1 — Overlay patch on clean images, then draw strict FPs

This method loads clean images, overlays the patch tensor at `(X_OFFSET, Y_OFFSET)`, runs YOLO, then saves only the strict FP visualization.


In [ ]:

def generate_overlay_patch_demos():
    out_dir = OUT_ROOT / 'overlay_patch_fp'
    out_dir.mkdir(parents=True, exist_ok=True)

    scan_paths = choose_image_paths(
        overlay_img_dir,
        n=N_DEMOS,
        preferred_names=DEMO_IMAGE_NAMES,
        max_scan=MAX_SCAN_IMAGES,
    )

    selected = []
    fallback = []

    print(f'Scanning up to {len(scan_paths)} clean images for overlay-patch demos...')

    for idx, img_path in enumerate(scan_paths, 1):
        base_tensor = load_image_tensor(img_path, device)
        patched_tensor, used_offset = overlay_patch(base_tensor, patch, X_OFFSET, Y_OFFSET)

        label_path = overlay_lab_dir / f'{img_path.stem}.txt'
        title = f'overlay patch | {img_path.name} | offset={used_offset}'
        vis_rgb, stats = predict_and_draw_fp(model, patched_tensor, label_path, class_names, title)

        row = {
            'method': 'overlay_patch',
            'image': img_path.name,
            'out_path': out_dir / f'{img_path.stem}_overlay_patch_strict_fp.png',
            **stats,
        }

        if stats['strict_fp'] > 0 or not REQUIRE_AT_LEAST_ONE_FP:
            Image.fromarray(vis_rgb).save(row['out_path'])
            selected.append(row)
            print(f"[{len(selected)}/{N_DEMOS}] {img_path.name}: GT={stats['gt']} preds={stats['preds']} TP={stats['tp']} strict_FP={stats['strict_fp']} ignored={stats['ignored']}")
        else:
            fallback.append((row, vis_rgb))

        if len(selected) >= N_DEMOS:
            break

    # If there are not enough images with FPs, fill the remaining slots with no-FP examples.
    if len(selected) < N_DEMOS:
        needed = N_DEMOS - len(selected)
        print(f'Only found {len(selected)} images with strict FP. Filling {needed} slot(s) with first no-FP examples.')
        for row, vis_rgb in fallback[:needed]:
            Image.fromarray(vis_rgb).save(row['out_path'])
            selected.append(row)

    grid_path = OUT_ROOT / 'overlay_patch_fp_grid.png'
    save_preview_grid(selected, grid_path, 'Overlay patch method: strict FP demos')

    return selected

overlay_rows = generate_overlay_patch_demos()
overlay_rows


## Method 2 — Load already-patched images from Drive, then draw strict FPs

This method does **not** overlay anything. It assumes the images inside `LOADED_PATCHED_ROOT` already contain the patch.

To make comparison easier, it first tries to use the same image names selected by the overlay method. If they are not present in the loaded patched split, it fills the list with available images from that split.


In [ ]:

def generate_loaded_patched_image_demos():
    out_dir = OUT_ROOT / 'loaded_patched_images_fp'
    out_dir.mkdir(parents=True, exist_ok=True)

    preferred_names = DEMO_IMAGE_NAMES
    if preferred_names is None and 'overlay_rows' in globals() and overlay_rows:
        preferred_names = [r['image'] for r in overlay_rows]

    scan_paths = choose_image_paths(
        loaded_img_dir,
        n=N_DEMOS,
        preferred_names=preferred_names,
        max_scan=MAX_SCAN_IMAGES,
    )

    selected = []
    fallback = []

    print(f'Scanning up to {len(scan_paths)} already-patched images for loaded-patched-image demos...')

    for idx, img_path in enumerate(scan_paths, 1):
        image_tensor = load_image_tensor(img_path, device)

        label_path = loaded_lab_dir / f'{img_path.stem}.txt'
        title = f'loaded patched image | {img_path.name}'
        vis_rgb, stats = predict_and_draw_fp(model, image_tensor, label_path, class_names, title)

        row = {
            'method': 'loaded_patched_images',
            'image': img_path.name,
            'out_path': out_dir / f'{img_path.stem}_loaded_patched_image_strict_fp.png',
            **stats,
        }

        if stats['strict_fp'] > 0 or not REQUIRE_AT_LEAST_ONE_FP:
            Image.fromarray(vis_rgb).save(row['out_path'])
            selected.append(row)
            print(f"[{len(selected)}/{N_DEMOS}] {img_path.name}: GT={stats['gt']} preds={stats['preds']} TP={stats['tp']} strict_FP={stats['strict_fp']} ignored={stats['ignored']}")
        else:
            fallback.append((row, vis_rgb))

        if len(selected) >= N_DEMOS:
            break

    if len(selected) < N_DEMOS:
        needed = N_DEMOS - len(selected)
        print(f'Only found {len(selected)} images with strict FP. Filling {needed} slot(s) with first no-FP examples.')
        for row, vis_rgb in fallback[:needed]:
            Image.fromarray(vis_rgb).save(row['out_path'])
            selected.append(row)

    grid_path = OUT_ROOT / 'loaded_patched_images_fp_grid.png'
    save_preview_grid(selected, grid_path, 'Loaded patched image method: strict FP demos')

    return selected

loaded_rows = generate_loaded_patched_image_demos()
loaded_rows


## Final summary table

In [ ]:
# ============================================================
# FIND EVERYTHING RELEVANT IN DRIVE
# No creation. No overlay. No prediction. Just search + print.
# ============================================================

from pathlib import Path
import os
import pandas as pd

ROOTS = [
    Path("/content/drive/MyDrive/isaac_scene1"),
    Path("/content/drive/MyDrive"),
]

IMAGE_NAME = "rgb_0027.png"
LABEL_NAME = IMAGE_NAME.replace(".png", ".txt")

# Use isaac_scene1 if it exists, otherwise search all MyDrive
SEARCH_ROOT = ROOTS[0] if ROOTS[0].exists() else ROOTS[1]

print("Searching inside:")
print(SEARCH_ROOT)
print()

# Folders to skip so Drive search does not become a funeral
SKIP_DIR_NAMES = {
    ".Trash",
    "Trash",
    "__pycache__",
    ".ipynb_checkpoints",
    ".git",
    "node_modules",
}

image_hits = []
label_hits = []
model_hits = []
patch_hits = []
yaml_hits = []
image_dirs = []
label_dirs = []

for dirpath, dirnames, filenames in os.walk(SEARCH_ROOT):
    dirnames[:] = [d for d in dirnames if d not in SKIP_DIR_NAMES]

    d = Path(dirpath)
    file_set = set(filenames)

    # exact rgb_0027 image
    if IMAGE_NAME in file_set:
        image_hits.append(d / IMAGE_NAME)

    # exact rgb_0027 label
    if LABEL_NAME in file_set:
        label_hits.append(d / LABEL_NAME)

    # useful folders
    if "images" in str(d).lower() or d.name.lower() == "images":
        png_count = sum(1 for f in filenames if f.lower().endswith((".png", ".jpg", ".jpeg")))
        if png_count > 0:
            image_dirs.append((d, png_count))

    if "labels" in str(d).lower() or d.name.lower() == "labels":
        txt_count = sum(1 for f in filenames if f.lower().endswith(".txt"))
        if txt_count > 0:
            label_dirs.append((d, txt_count))

    # models / patches / yamls
    for f in filenames:
        low = f.lower()
        p = d / f

        if low == "best.pt" or low.endswith(".pt"):
            if "weight" in str(p).lower() or "yolo_runs" in str(p).lower():
                model_hits.append(p)
            elif "patch" in low or "bill" in low or "adv" in low:
                patch_hits.append(p)
            else:
                model_hits.append(p)

        if low.endswith((".yaml", ".yml")):
            yaml_hits.append(p)

# Sort nicely
image_hits = sorted(set(image_hits))
label_hits = sorted(set(label_hits))
model_hits = sorted(set(model_hits))
patch_hits = sorted(set(patch_hits))
yaml_hits = sorted(set(yaml_hits))
image_dirs = sorted(set(image_dirs), key=lambda x: str(x[0]))
label_dirs = sorted(set(label_dirs), key=lambda x: str(x[0]))

print("=" * 80)
print(f"EXACT IMAGE HITS FOR {IMAGE_NAME}: {len(image_hits)}")
print("=" * 80)
for i, p in enumerate(image_hits):
    print(f"[IMG {i}] {p}")

print()
print("=" * 80)
print(f"EXACT LABEL HITS FOR {LABEL_NAME}: {len(label_hits)}")
print("=" * 80)
for i, p in enumerate(label_hits):
    print(f"[LBL {i}] {p}")

print()
print("=" * 80)
print(f"MODEL / .PT HITS: {len(model_hits)}")
print("=" * 80)
for i, p in enumerate(model_hits[:100]):
    print(f"[PT {i}] {p}")

print()
print("=" * 80)
print(f"PATCH-LIKE .PT HITS: {len(patch_hits)}")
print("=" * 80)
for i, p in enumerate(patch_hits[:100]):
    print(f"[PATCH {i}] {p}")

print()
print("=" * 80)
print(f"YAML HITS: {len(yaml_hits)}")
print("=" * 80)
for i, p in enumerate(yaml_hits[:100]):
    print(f"[YAML {i}] {p}")

print()
print("=" * 80)
print(f"IMAGE DIRECTORIES FOUND: {len(image_dirs)}")
print("=" * 80)
for i, (p, n) in enumerate(image_dirs[:200]):
    print(f"[IMG_DIR {i}] {p}  --> {n} image files")

print()
print("=" * 80)
print(f"LABEL DIRECTORIES FOUND: {len(label_dirs)}")
print("=" * 80)
for i, (p, n) in enumerate(label_dirs[:200]):
    print(f"[LBL_DIR {i}] {p}  --> {n} txt files")

# Create a clean dataframe summary too
rows = []

for p in image_hits:
    rows.append({
        "type": "exact_image_rgb_0027",
        "path": str(p),
        "exists": p.exists()
    })

for p in label_hits:
    rows.append({
        "type": "exact_label_rgb_0027",
        "path": str(p),
        "exists": p.exists()
    })

for p in model_hits:
    rows.append({
        "type": "model_or_pt",
        "path": str(p),
        "exists": p.exists()
    })

for p in patch_hits:
    rows.append({
        "type": "patch_like_pt",
        "path": str(p),
        "exists": p.exists()
    })

for p in yaml_hits:
    rows.append({
        "type": "yaml",
        "path": str(p),
        "exists": p.exists()
    })

df = pd.DataFrame(rows)

print()
print("=" * 80)
print("DATAFRAME SUMMARY")
print("=" * 80)

display(df)

# Also save the search result
out_csv = Path("/content/drive/MyDrive/isaac_scene1_drive_search_results.csv")
df.to_csv(out_csv, index=False)

print()
print("Saved search summary to:")
print(out_csv)

In [ ]:
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# ============================================================
# KEEP ALL FP CANDIDATES
# No title bar.
# No discarding.
# No overlay creation.
# No patched image creation.
# ============================================================

IMAGE_NAME = "rgb_0027.png"

MODEL_PATH = Path("/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt")

CLEAN_IMG_PATH = Path("/content/drive/MyDrive/isaac_scene1/clean/yolo_filtered/images/test") / IMAGE_NAME
DIGITAL_IMG_PATH = Path("/content/drive/MyDrive/isaac_scene1/transf_Patch1/images/test") / IMAGE_NAME
ACTUAL_IMG_PATH = Path("/content/drive/MyDrive/isaac_scene1/yolo_patched/images/test") / IMAGE_NAME

OUT_DIR = Path("/content/drive/MyDrive/isaac_rgb0027_keep_all_fp_no_top")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMGSZ = 640
CONF = 0.25
IOU = 0.6
MAX_DET = 1000

# Lower = stricter matching to clean image.
# If patched detection does not match clean detection, we keep it as FP.
MATCH_IOU = 0.30

def load_img_640(path):
    img = Image.open(path).convert("RGB")
    img = img.resize((IMGSZ, IMGSZ))
    return np.array(img).astype(np.uint8)

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)

    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0

def cls_name(names, c):
    c = int(c)
    if isinstance(names, dict):
        return names.get(c, str(c))
    return names[c] if 0 <= c < len(names) else str(c)

def predict_boxes(model, img_rgb):
    result = model.predict(
        source=img_rgb,
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        max_det=MAX_DET,
        verbose=False,
    )[0]

    boxes = []

    if result.boxes is None or len(result.boxes) == 0:
        return boxes

    pred_cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    pred_conf = result.boxes.conf.detach().cpu().numpy()
    pred_xyxy = result.boxes.xyxy.detach().cpu().numpy()

    for i in range(len(pred_cls)):
        boxes.append({
            "cls": int(pred_cls[i]),
            "conf": float(pred_conf[i]),
            "xyxy": pred_xyxy[i].tolist(),
        })

    return boxes

def get_all_fp_candidates(clean_boxes, patched_boxes):
    """
    Keeps ALL patched detections that do not match a clean detection.
    Does NOT discard anything into ignored.
    """
    fps = []

    for p in patched_boxes:
        pbox = p["xyxy"]

        best_clean_iou = 0.0
        best_clean_cls = None

        for c in clean_boxes:
            v = iou_xyxy(pbox, c["xyxy"])
            if v > best_clean_iou:
                best_clean_iou = v
                best_clean_cls = c["cls"]

        # Only remove detections that already existed in the clean image.
        # Everything else is kept as FP.
        if best_clean_iou < MATCH_IOU:
            p["best_clean_iou"] = best_clean_iou
            p["best_clean_cls"] = best_clean_cls
            fps.append(p)

    return fps

def draw_all_fps_no_top(img_rgb, fps, names):
    out = cv2.cvtColor(img_rgb.copy(), cv2.COLOR_RGB2BGR)

    for item in fps:
        cls_id = item["cls"]
        conf = item["conf"]
        box = item["xyxy"]
        best_iou = item["best_clean_iou"]

        x1, y1, x2, y2 = map(int, box)

        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

        label = f"FP {cls_name(names, cls_id)} {conf:.2f} cleanIoU {best_iou:.2f}"

        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 0.42
        thickness = 1

        (tw, th), _ = cv2.getTextSize(label, font, scale, thickness)

        tx = max(0, min(x1, IMGSZ - tw - 6))
        ty = max(th + 8, y1)

        cv2.rectangle(
            out,
            (tx, ty - th - 6),
            (tx + tw + 4, ty),
            (0, 0, 255),
            -1,
        )

        cv2.putText(
            out,
            label,
            (tx + 2, ty - 4),
            font,
            scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA,
        )

    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

# ============================================================
# RUN
# ============================================================

model = YOLO(str(MODEL_PATH))

clean_img = load_img_640(CLEAN_IMG_PATH)
digital_img = load_img_640(DIGITAL_IMG_PATH)
actual_img = load_img_640(ACTUAL_IMG_PATH)

clean_boxes = predict_boxes(model, clean_img)
digital_boxes = predict_boxes(model, digital_img)
actual_boxes = predict_boxes(model, actual_img)

digital_fps = get_all_fp_candidates(clean_boxes, digital_boxes)
actual_fps = get_all_fp_candidates(clean_boxes, actual_boxes)

digital_vis = draw_all_fps_no_top(digital_img, digital_fps, model.names)
actual_vis = draw_all_fps_no_top(actual_img, actual_fps, model.names)

digital_out = OUT_DIR / "rgb_0027_digital_overlay_KEEP_ALL_FP_no_top.png"
actual_out = OUT_DIR / "rgb_0027_actually_placed_KEEP_ALL_FP_no_top.png"
compare_out = OUT_DIR / "rgb_0027_KEEP_ALL_FP_compare_no_top.png"

Image.fromarray(digital_vis).save(digital_out)
Image.fromarray(actual_vis).save(actual_out)
Image.fromarray(np.concatenate([digital_vis, actual_vis], axis=1)).save(compare_out)

print("Clean detections:", len(clean_boxes))
print("Digital detections:", len(digital_boxes))
print("Digital FP kept:", len(digital_fps))
print("Actual detections:", len(actual_boxes))
print("Actual FP kept:", len(actual_fps))

print("\nSaved:")
print(digital_out)
print(actual_out)
print(compare_out)

plt.figure(figsize=(18, 8))
plt.imshow(np.concatenate([digital_vis, actual_vis], axis=1))
plt.axis("off")
plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# ============================================================
# STRICT FP DEMOS LIKE YOUR generate_loaded_patched_image_demos()
# Loads existing images only.
# No overlay creation.
# No patched-image creation.
# ============================================================

ROOT = Path("/content/drive/MyDrive/isaac_scene1")

MODEL_PATH = ROOT / "yolo_runs/yolov8n_isaac_scene12/weights/best.pt"

DIGITAL_ROOT = ROOT / "transf_Patch1"
ACTUAL_ROOT = ROOT / "yolo_patched"

OUT_ROOT = ROOT / "fp_demos_like_original_method"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

N_DEMOS = 5
MAX_SCAN_IMAGES = 300
REQUIRE_AT_LEAST_ONE_FP = True

# Put rgb_0027 first, then scan the rest.
DEMO_IMAGE_NAMES = ["rgb_0027.png"]

IMGSZ = 640
CONF = 0.25
IOU = 0.6
MAX_DET = 1000

GT_IOU_MATCH = 0.50
IGNORE_IF_OVERLAPS_ANY_GT_IOU = 0.30

# If True: every unmatched prediction is drawn as FP.
# If False: exact strict method: overlapping wrong detections go to ignored.
KEEP_ALL_UNMATCHED_AS_FP = False

# No black title bar on top.
DRAW_TOP_TITLE = False

print("MODEL:", MODEL_PATH, MODEL_PATH.exists())
print("DIGITAL ROOT:", DIGITAL_ROOT, DIGITAL_ROOT.exists())
print("ACTUAL ROOT:", ACTUAL_ROOT, ACTUAL_ROOT.exists())

for p, name in [
    (MODEL_PATH, "model"),
    (DIGITAL_ROOT / "images/test", "digital images/test"),
    (DIGITAL_ROOT / "labels/test", "digital labels/test"),
    (ACTUAL_ROOT / "images/test", "actual images/test"),
    (ACTUAL_ROOT / "labels/test", "actual labels/test"),
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {name}: {p}")

model = YOLO(str(MODEL_PATH))
class_names = model.names

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(device)

def load_image_array(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((IMGSZ, IMGSZ))
    return np.array(img).astype(np.uint8)

def read_yolo_label_file(label_path):
    rows = []

    if not label_path.exists():
        return rows

    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            cls = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            rows.append((cls, x, y, w, h))

    return rows

def xywhn_to_xyxy(x, y, w, h):
    return [
        (x - w / 2) * IMGSZ,
        (y - h / 2) * IMGSZ,
        (x + w / 2) * IMGSZ,
        (y + h / 2) * IMGSZ,
    ]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)

    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0

def name_of_class(names, cls_id):
    if cls_id is None:
        return "none"

    cls_id = int(cls_id)

    if isinstance(names, dict):
        return names.get(cls_id, str(cls_id))

    if 0 <= cls_id < len(names):
        return names[cls_id]

    return str(cls_id)

def get_fp_tp_ignored(result, label_path):
    labels = read_yolo_label_file(label_path)

    gt_by_class = {}
    all_gt = []

    for cls, x, y, w, h in labels:
        box = xywhn_to_xyxy(x, y, w, h)
        gt_by_class.setdefault(cls, []).append(box)
        all_gt.append((cls, box))

    tp = []
    strict_fp = []
    ignored = []

    if result.boxes is None or len(result.boxes) == 0:
        return strict_fp, tp, ignored, len(labels), 0

    pred_cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    pred_conf = result.boxes.conf.detach().cpu().numpy()
    pred_xyxy = result.boxes.xyxy.detach().cpu().numpy()

    preds = []
    for i in range(len(pred_cls)):
        preds.append({
            "cls": int(pred_cls[i]),
            "conf": float(pred_conf[i]),
            "box": pred_xyxy[i].tolist(),
        })

    preds.sort(key=lambda z: z["conf"], reverse=True)

    remaining_gt = {c: list(boxes) for c, boxes in gt_by_class.items()}
    unmatched = []

    # First pass: match true positives by same class and IoU.
    for pred in preds:
        pc = pred["cls"]
        pbox = pred["box"]

        best_same_iou = 0.0
        best_j = None

        same_class_gts = remaining_gt.get(pc, [])

        for j, gt_box in enumerate(same_class_gts):
            v = iou_xyxy(pbox, gt_box)
            if v > best_same_iou:
                best_same_iou = v
                best_j = j

        if best_j is not None and best_same_iou >= GT_IOU_MATCH:
            tp.append({
                **pred,
                "matched_iou": best_same_iou,
            })
            same_class_gts.pop(best_j)
            remaining_gt[pc] = same_class_gts
        else:
            unmatched.append(pred)

    # Second pass: classify unmatched predictions.
    for pred in unmatched:
        pbox = pred["box"]

        best_any_iou = 0.0
        best_any_cls = None

        for gt_cls, gt_box in all_gt:
            v = iou_xyxy(pbox, gt_box)
            if v > best_any_iou:
                best_any_iou = v
                best_any_cls = gt_cls

        pred["best_any_iou"] = best_any_iou
        pred["best_any_cls"] = best_any_cls

        if KEEP_ALL_UNMATCHED_AS_FP:
            strict_fp.append(pred)
        else:
            if best_any_iou >= IGNORE_IF_OVERLAPS_ANY_GT_IOU:
                ignored.append(pred)
            else:
                strict_fp.append(pred)

    return strict_fp, tp, ignored, len(labels), len(preds)

def draw_fp_only(img_rgb, strict_fp, class_names, title):
    out = cv2.cvtColor(img_rgb.copy(), cv2.COLOR_RGB2BGR)

    if DRAW_TOP_TITLE:
        cv2.rectangle(out, (0, 0), (IMGSZ, 30), (0, 0, 0), -1)
        cv2.putText(
            out,
            title,
            (8, 21),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.52,
            (255, 255, 255),
            1,
            cv2.LINE_AA,
        )

    for pred in strict_fp:
        cls_id = pred["cls"]
        conf = pred["conf"]
        box = pred["box"]
        best_iou = pred.get("best_any_iou", 0.0)
        best_cls = pred.get("best_any_cls", None)

        x1, y1, x2, y2 = map(int, box)

        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

        label = (
            f"FP {name_of_class(class_names, cls_id)} "
            f"{conf:.2f} IoU {best_iou:.2f} "
            f"vs {name_of_class(class_names, best_cls)}"
        )

        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 0.42
        thickness = 1

        (tw, th), _ = cv2.getTextSize(label, font, scale, thickness)

        tx = max(0, min(x1, IMGSZ - tw - 6))

        if DRAW_TOP_TITLE:
            ty = max(34 + th, y1)
        else:
            ty = max(th + 8, y1)

        cv2.rectangle(
            out,
            (tx, ty - th - 6),
            (tx + tw + 4, ty),
            (0, 0, 255),
            -1,
        )

        cv2.putText(
            out,
            label,
            (tx + 2, ty - 4),
            font,
            scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA,
        )

    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

def predict_and_draw_fp(model, img_path, label_path, class_names, title):
    img_rgb = load_image_array(img_path)

    result = model.predict(
        source=img_rgb,
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        max_det=MAX_DET,
        device=device,
        verbose=False,
    )[0]

    strict_fp, tp, ignored, gt_count, pred_count = get_fp_tp_ignored(result, label_path)

    vis_rgb = draw_fp_only(
        img_rgb,
        strict_fp,
        class_names,
        f"{title} | FP={len(strict_fp)}",
    )

    stats = {
        "gt": gt_count,
        "preds": pred_count,
        "tp": len(tp),
        "strict_fp": len(strict_fp),
        "ignored": len(ignored),
    }

    return vis_rgb, stats

def choose_image_paths(img_dir, n, preferred_names=None, max_scan=300):
    all_imgs = sorted(
        list(img_dir.glob("*.png")) +
        list(img_dir.glob("*.jpg")) +
        list(img_dir.glob("*.jpeg"))
    )

    chosen = []

    if preferred_names:
        name_to_path = {p.name: p for p in all_imgs}
        for name in preferred_names:
            if name in name_to_path:
                chosen.append(name_to_path[name])

    for p in all_imgs:
        if p not in chosen:
            chosen.append(p)

    return chosen[:max_scan]

def save_preview_grid(rows, grid_path, title):
    if not rows:
        print(f"No rows for grid: {title}")
        return

    imgs = []
    captions = []

    for row in rows:
        img = Image.open(row["out_path"]).convert("RGB")
        imgs.append(np.array(img))
        captions.append(
            f"{row['image']}\n"
            f"GT={row['gt']} pred={row['preds']} "
            f"TP={row['tp']} FP={row['strict_fp']} ign={row['ignored']}"
        )

    cols = min(5, len(imgs))
    rows_n = int(np.ceil(len(imgs) / cols))

    plt.figure(figsize=(cols * 5, rows_n * 5))

    for i, img in enumerate(imgs):
        plt.subplot(rows_n, cols, i + 1)
        plt.imshow(img)
        plt.title(captions[i], fontsize=9)
        plt.axis("off")

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(grid_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved grid:", grid_path)

def generate_existing_image_fp_demos(method_name, dataset_root):
    img_dir = dataset_root / "images/test"
    lab_dir = dataset_root / "labels/test"

    out_dir = OUT_ROOT / f"{method_name}_fp"
    out_dir.mkdir(parents=True, exist_ok=True)

    scan_paths = choose_image_paths(
        img_dir,
        n=N_DEMOS,
        preferred_names=DEMO_IMAGE_NAMES,
        max_scan=MAX_SCAN_IMAGES,
    )

    selected = []
    fallback = []

    print()
    print("=" * 90)
    print(f"Scanning up to {len(scan_paths)} images for {method_name} demos...")
    print("Images:", img_dir)
    print("Labels:", lab_dir)
    print("=" * 90)

    for idx, img_path in enumerate(scan_paths, 1):
        label_path = lab_dir / f"{img_path.stem}.txt"
        title = f"{method_name} | {img_path.name}"

        vis_rgb, stats = predict_and_draw_fp(
            model,
            img_path,
            label_path,
            class_names,
            title,
        )

        row = {
            "method": method_name,
            "image": img_path.name,
            "out_path": out_dir / f"{img_path.stem}_{method_name}_strict_fp.png",
            **stats,
        }

        if stats["strict_fp"] > 0 or not REQUIRE_AT_LEAST_ONE_FP:
            Image.fromarray(vis_rgb).save(row["out_path"])
            selected.append(row)

            print(
                f"[{len(selected)}/{N_DEMOS}] {img_path.name}: "
                f"GT={stats['gt']} preds={stats['preds']} "
                f"TP={stats['tp']} strict_FP={stats['strict_fp']} "
                f"ignored={stats['ignored']}"
            )
        else:
            fallback.append((row, vis_rgb))

        if len(selected) >= N_DEMOS:
            break

    if len(selected) < N_DEMOS:
        needed = N_DEMOS - len(selected)
        print(
            f"Only found {len(selected)} images with strict FP. "
            f"Filling {needed} slot(s) with first no-FP examples."
        )

        for row, vis_rgb in fallback[:needed]:
            Image.fromarray(vis_rgb).save(row["out_path"])
            selected.append(row)

    grid_path = OUT_ROOT / f"{method_name}_fp_grid.png"
    save_preview_grid(
        selected,
        grid_path,
        f"{method_name}: strict FP demos",
    )

    return selected

digital_rows = generate_existing_image_fp_demos(
    method_name="digital_overlay_transf_Patch1",
    dataset_root=DIGITAL_ROOT,
)

actual_rows = generate_existing_image_fp_demos(
    method_name="actually_placed_yolo_patched",
    dataset_root=ACTUAL_ROOT,
)

digital_rows, actual_rows

In [ ]:
# ============================================================
# STOP IGNORING
# Every unmatched prediction becomes FP.
# ignored is always 0.
# ============================================================

KEEP_ALL_UNMATCHED_AS_FP = True

def get_fp_tp_ignored(result, label_path):
    labels = read_yolo_label_file(label_path)

    gt_by_class = {}

    for cls, x, y, w, h in labels:
        box = xywhn_to_xyxy(x, y, w, h)
        gt_by_class.setdefault(cls, []).append(box)

    tp = []
    strict_fp = []
    ignored = []   # stays empty forever

    if result.boxes is None or len(result.boxes) == 0:
        return strict_fp, tp, ignored, len(labels), 0

    pred_cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    pred_conf = result.boxes.conf.detach().cpu().numpy()
    pred_xyxy = result.boxes.xyxy.detach().cpu().numpy()

    preds = []

    for i in range(len(pred_cls)):
        preds.append({
            "cls": int(pred_cls[i]),
            "conf": float(pred_conf[i]),
            "box": pred_xyxy[i].tolist(),
        })

    preds.sort(key=lambda z: z["conf"], reverse=True)

    remaining_gt = {c: list(boxes) for c, boxes in gt_by_class.items()}

    for pred in preds:
        pc = pred["cls"]
        pbox = pred["box"]

        best_same_iou = 0.0
        best_j = None

        same_class_gts = remaining_gt.get(pc, [])

        for j, gt_box in enumerate(same_class_gts):
            v = iou_xyxy(pbox, gt_box)
            if v > best_same_iou:
                best_same_iou = v
                best_j = j

        if best_j is not None and best_same_iou >= GT_IOU_MATCH:
            tp.append({
                **pred,
                "matched_iou": best_same_iou,
            })
            same_class_gts.pop(best_j)
            remaining_gt[pc] = same_class_gts
        else:
            # No ignoring.
            # If it is not a same-class TP, it is FP.
            pred["best_any_iou"] = best_same_iou
            pred["best_any_cls"] = pc
            strict_fp.append(pred)

    return strict_fp, tp, ignored, len(labels), len(preds)


# Use a fresh output folder so old ignored results do not mix with new ones
OUT_ROOT = ROOT / "fp_demos_NO_IGNORE_all_unmatched_are_fp"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

digital_rows = generate_existing_image_fp_demos(
    method_name="digital_overlay_transf_Patch1_NO_IGNORE",
    dataset_root=DIGITAL_ROOT,
)

actual_rows = generate_existing_image_fp_demos(
    method_name="actually_placed_yolo_patched_NO_IGNORE",
    dataset_root=ACTUAL_ROOT,
)

digital_rows, actual_rows

In [ ]:
# ============================================================
# RESTORE MANY-FP BEHAVIOR
# Low confidence + no ignored bucket.
# Same images, same labels, same drawing style.
# ============================================================

CONF = 0.1                  # important: was 0.25, which hides many FPs
KEEP_ALL_UNMATCHED_AS_FP = True
DRAW_TOP_TITLE = False        # keep no black title bar

def get_fp_tp_ignored(result, label_path):
    labels = read_yolo_label_file(label_path)

    gt_by_class = {}

    for cls, x, y, w, h in labels:
        box = xywhn_to_xyxy(x, y, w, h)
        gt_by_class.setdefault(cls, []).append(box)

    tp = []
    strict_fp = []
    ignored = []   # always empty now

    if result.boxes is None or len(result.boxes) == 0:
        return strict_fp, tp, ignored, len(labels), 0

    pred_cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    pred_conf = result.boxes.conf.detach().cpu().numpy()
    pred_xyxy = result.boxes.xyxy.detach().cpu().numpy()

    preds = []

    for i in range(len(pred_cls)):
        preds.append({
            "cls": int(pred_cls[i]),
            "conf": float(pred_conf[i]),
            "box": pred_xyxy[i].tolist(),
        })

    preds.sort(key=lambda z: z["conf"], reverse=True)

    remaining_gt = {c: list(boxes) for c, boxes in gt_by_class.items()}

    for pred in preds:
        pc = pred["cls"]
        pbox = pred["box"]

        best_same_iou = 0.0
        best_j = None

        same_class_gts = remaining_gt.get(pc, [])

        for j, gt_box in enumerate(same_class_gts):
            v = iou_xyxy(pbox, gt_box)
            if v > best_same_iou:
                best_same_iou = v
                best_j = j

        if best_j is not None and best_same_iou >= GT_IOU_MATCH:
            tp.append({
                **pred,
                "matched_iou": best_same_iou,
            })
            same_class_gts.pop(best_j)
            remaining_gt[pc] = same_class_gts
        else:
            # No ignored bucket.
            # If it is not a same-class TP, it is FP.
            pred["best_any_iou"] = best_same_iou
            pred["best_any_cls"] = pc
            strict_fp.append(pred)

    return strict_fp, tp, ignored, len(labels), len(preds)


OUT_ROOT = ROOT / "fp_demos_many_fp_like_before_conf0001_no_ignore"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

digital_rows = generate_existing_image_fp_demos(
    method_name="digital_overlay_transf_Patch1_MANY_FP",
    dataset_root=DIGITAL_ROOT,
)

actual_rows = generate_existing_image_fp_demos(
    method_name="actually_placed_yolo_patched_MANY_FP",
    dataset_root=ACTUAL_ROOT,
)

digital_rows, actual_rows

In [ ]:
from pathlib import Path
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# ============================================================
# FP VISUALIZATION USING evaluation_ab.py LOGIC
# No overlay creation.
# No patched image creation.
# FP = unmatched prediction vs same-class GT, exactly like count_fp_at().
# ============================================================

ROOT = Path("/content/drive/MyDrive/isaac_scene1")

MODEL_PATH = ROOT / "yolo_runs/yolov8n_isaac_scene12/weights/best.pt"

DATASET_ROOTS = {
    "digital_overlay_transf_Patch1": ROOT / "transf_Patch1",
    "actually_placed_yolo_patched": ROOT / "yolo_patched",
}

SPLIT = "test"

OUT_ROOT = ROOT / "fp_visuals_eval_ab_logic"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

IMGSZ = 640
CONF_MIN = 0.001
IOU = 0.6
GT_IOU_MATCH = 0.5
MAX_DET = 1000

# For drawing.
# Use 0.3 for clean visuals.
# Use CONF_MIN if you want the exact fp_total_conf_ge_conf_min behavior.
DRAW_THR = 0.3

N_DEMOS = 5
MAX_SCAN_IMAGES = 300
REQUIRE_AT_LEAST_ONE_FP = True

# Put rgb_0027 first.
DEMO_IMAGE_NAMES = ["rgb_0027.png"]

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("MODEL:", MODEL_PATH, MODEL_PATH.exists())
for name, root in DATASET_ROOTS.items():
    print(name, root, root.exists())
    print("  images:", (root / "images" / SPLIT), (root / "images" / SPLIT).exists())
    print("  labels:", (root / "labels" / SPLIT), (root / "labels" / SPLIT).exists())

if not MODEL_PATH.exists():
    raise FileNotFoundError(MODEL_PATH)

model = YOLO(str(MODEL_PATH))


def load_img_640(path):
    img = Image.open(path).convert("RGB")
    img = img.resize((IMGSZ, IMGSZ))
    return np.array(img).astype(np.uint8)


def read_yolo_gt_labels(txt_path):
    if not txt_path.exists():
        return []

    out = []
    for ln in txt_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue

    return out


def xywhn_to_xyxy(x, y, w, h):
    x1 = x - w / 2.0
    y1 = y - h / 2.0
    x2 = x + w / 2.0
    y2 = y + h / 2.0
    return x1, y1, x2, y2


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)

    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0


def class_name(names, c):
    c = int(c)

    if isinstance(names, dict):
        return names.get(c, str(c))

    if 0 <= c < len(names):
        return names[c]

    return str(c)


def predict_boxes_eval_ab_style(model, img_rgb):
    result = model.predict(
        source=img_rgb,
        imgsz=IMGSZ,
        conf=CONF_MIN,
        iou=IOU,
        max_det=MAX_DET,
        device="cpu",
        verbose=False,
    )[0]

    boxes = result.boxes

    preds = []

    if boxes is None or len(boxes) == 0:
        return preds

    pred_cls = boxes.cls.detach().cpu().numpy().astype(int)
    pred_xywhn = boxes.xywhn.detach().cpu().numpy().astype(np.float32)
    pred_conf = boxes.conf.detach().cpu().numpy().astype(np.float32)

    for i in range(len(pred_cls)):
        c = int(pred_cls[i])
        x, y, w, h = pred_xywhn[i].tolist()

        preds.append({
            "cls": c,
            "box_norm": xywhn_to_xyxy(x, y, w, h),
            "conf": float(pred_conf[i]),
        })

    return preds


def find_fps_eval_ab_logic(preds, gt_rows, thr):
    """
    Same logic as evaluation_ab.py count_fp_at(thr):

    1. Keep predictions with conf >= thr.
    2. Sort by confidence descending.
    3. For each prediction, search remaining GT boxes of the SAME class.
    4. If best IoU >= GT_IOU_MATCH -> TP and consume that GT.
    5. Else -> FP.
    """

    pf = [p for p in preds if p["conf"] >= thr]
    pf.sort(key=lambda z: z["conf"], reverse=True)

    gt_by_class = {}
    for c, x, y, w, h in gt_rows:
        gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h))

    remaining = {c: list(lst) for c, lst in gt_by_class.items()}

    tp = []
    fp = []

    for pred in pf:
        c = pred["cls"]
        pbox = pred["box_norm"]

        best_iou = 0.0
        best_j = None

        gtl = remaining.get(c, [])

        for j, gbox in enumerate(gtl):
            iouv = iou_xyxy(pbox, gbox)
            if iouv > best_iou:
                best_iou = iouv
                best_j = j

        if best_j is not None and best_iou >= GT_IOU_MATCH:
            pred["matched_iou"] = best_iou
            tp.append(pred)
            gtl.pop(best_j)
            remaining[c] = gtl
        else:
            pred["best_same_class_iou"] = best_iou
            fp.append(pred)

    return tp, fp, len(pf)


def norm_box_to_pixel(box_norm):
    x1, y1, x2, y2 = box_norm
    return [
        int(round(x1 * IMGSZ)),
        int(round(y1 * IMGSZ)),
        int(round(x2 * IMGSZ)),
        int(round(y2 * IMGSZ)),
    ]


def draw_fps_no_top_text(img_rgb, fps, names):
    out = cv2.cvtColor(img_rgb.copy(), cv2.COLOR_RGB2BGR)

    for pred in fps:
        cls_id = pred["cls"]
        conf = pred["conf"]
        best_iou = pred.get("best_same_class_iou", 0.0)

        x1, y1, x2, y2 = norm_box_to_pixel(pred["box_norm"])

        x1 = max(0, min(IMGSZ - 1, x1))
        y1 = max(0, min(IMGSZ - 1, y1))
        x2 = max(0, min(IMGSZ - 1, x2))
        y2 = max(0, min(IMGSZ - 1, y2))

        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

        label = f"FP {class_name(names, cls_id)} {conf:.2f} IoU {best_iou:.2f}"

        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 0.42
        thickness = 1

        (tw, th), _ = cv2.getTextSize(label, font, scale, thickness)

        tx = max(0, min(x1, IMGSZ - tw - 6))
        ty = max(th + 8, y1)

        cv2.rectangle(
            out,
            (tx, ty - th - 6),
            (tx + tw + 4, ty),
            (0, 0, 255),
            -1,
        )

        cv2.putText(
            out,
            label,
            (tx + 2, ty - 4),
            font,
            scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA,
        )

    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)


def choose_image_paths(img_dir, preferred_names=None, max_scan=300):
    all_imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS])

    selected = []

    if preferred_names:
        lookup = {p.name: p for p in all_imgs}
        for name in preferred_names:
            if name in lookup:
                selected.append(lookup[name])

    for p in all_imgs:
        if p not in selected:
            selected.append(p)

    return selected[:max_scan]


def make_grid(saved_paths, grid_path, title):
    if not saved_paths:
        print("No images for grid:", title)
        return

    imgs = [np.array(Image.open(p).convert("RGB")) for p in saved_paths]

    cols = min(5, len(imgs))
    rows = int(np.ceil(len(imgs) / cols))

    plt.figure(figsize=(cols * 5, rows * 5))

    for i, img in enumerate(imgs):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(grid_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved grid:", grid_path)


def generate_fp_demos_for_dataset(method_name, dataset_root):
    img_dir = dataset_root / "images" / SPLIT
    lab_dir = dataset_root / "labels" / SPLIT

    if not img_dir.exists():
        raise FileNotFoundError(f"Missing image dir: {img_dir}")
    if not lab_dir.exists():
        raise FileNotFoundError(f"Missing label dir: {lab_dir}")

    out_dir = OUT_ROOT / method_name
    out_dir.mkdir(parents=True, exist_ok=True)

    scan_paths = choose_image_paths(
        img_dir,
        preferred_names=DEMO_IMAGE_NAMES,
        max_scan=MAX_SCAN_IMAGES,
    )

    selected_rows = []
    fallback_rows = []

    print()
    print("=" * 90)
    print(f"Scanning {method_name}")
    print("Images:", img_dir)
    print("Labels:", lab_dir)
    print("=" * 90)

    for img_path in scan_paths:
        label_path = lab_dir / f"{img_path.stem}.txt"

        img_rgb = load_img_640(img_path)
        gt_rows = read_yolo_gt_labels(label_path)
        preds = predict_boxes_eval_ab_style(model, img_rgb)

        tp, fp, n_pred_at_thr = find_fps_eval_ab_logic(
            preds=preds,
            gt_rows=gt_rows,
            thr=DRAW_THR,
        )

        vis = draw_fps_no_top_text(img_rgb, fp, model.names)

        out_path = out_dir / f"{img_path.stem}_{method_name}_eval_ab_FP_thr_{DRAW_THR}.png"

        row = {
            "method": method_name,
            "image": img_path.name,
            "image_path": str(img_path),
            "label_path": str(label_path),
            "out_path": str(out_path),
            "thr": DRAW_THR,
            "gt": len(gt_rows),
            "preds_conf_ge_thr": n_pred_at_thr,
            "tp": len(tp),
            "fp": len(fp),
        }

        if len(fp) > 0 or not REQUIRE_AT_LEAST_ONE_FP:
            Image.fromarray(vis).save(out_path)
            selected_rows.append(row)

            print(
                f"[{len(selected_rows)}/{N_DEMOS}] {img_path.name}: "
                f"GT={row['gt']} preds@{DRAW_THR}={row['preds_conf_ge_thr']} "
                f"TP={row['tp']} FP={row['fp']}"
            )
        else:
            fallback_rows.append((row, vis, out_path))

        if len(selected_rows) >= N_DEMOS:
            break

    if len(selected_rows) < N_DEMOS:
        needed = N_DEMOS - len(selected_rows)
        print(f"Only found {len(selected_rows)} images with FP. Filling {needed} no-FP examples.")

        for row, vis, out_path in fallback_rows[:needed]:
            Image.fromarray(vis).save(out_path)
            selected_rows.append(row)

    saved_paths = [Path(r["out_path"]) for r in selected_rows]

    grid_path = OUT_ROOT / f"{method_name}_eval_ab_FP_grid_thr_{DRAW_THR}.png"
    make_grid(saved_paths, grid_path, f"{method_name}: eval_ab FP logic, thr={DRAW_THR}")

    return selected_rows


all_rows = []

for method_name, dataset_root in DATASET_ROOTS.items():
    rows = generate_fp_demos_for_dataset(method_name, dataset_root)
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)

csv_path = OUT_ROOT / f"fp_rows_eval_ab_logic_thr_{DRAW_THR}.csv"
df.to_csv(csv_path, index=False)

print()
print("Saved CSV:")
print(csv_path)

display(df)